# Covariance Kernel Estimation and Pixel Correlation

This notebook demonstrates the full pipeline of estimating the pixel-to-pixel
covariance kernel from JWST difference images and illustrating the correlated
noise structure.

**Data:** `example3_f444w_diff.fits` + `example3_f444w_mask.fits` (F444W, 60 mas/pixel)

## Pipeline overview

1. **Source masking** — detect and mask all real astrophysical sources
2. **Find source-free regions** — locate the largest square patch with no sources
3. **Estimate covariance kernel** — compute the autocorrelation function (ACF) of the sky patch
4. **Apply window function** — taper the kernel edges to prevent Fourier ringing
5. **Compute power spectrum** — encode the covariance structure for the likelihood
6. **Demonstrate whitening** — show how dividing by the power spectrum removes correlations


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from astropy.io import fits
from astropy.stats import sigma_clipped_stats

import jwst_psfmc as jpm

mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('font', size=12)

print(f'jwst_psfmc version: {jpm.__version__}')

## 1. Load data

Load the difference image and source mask for the F444W epoch.


In [ ]:
# -- Load example data ---------------------------------------------------
diff = fits.getdata('data/example3_f444w_diff.fits').astype(np.float64)
mask = fits.getdata('data/example3_f444w_mask.fits').astype(bool)

print(f'Difference image shape: {diff.shape}')
print(f'Mask shape: {mask.shape}')
print(f'Source coverage: {mask.sum() / mask.size:.1%}')

## 2. Source masking

The mask file marks pixels containing real astrophysical emission. For covariance
kernel estimation, we need to exclude these regions -- the kernel must be computed
from **source-free sky only**.

We use `jwst_psfmc.get_source_mask()` to generate masks from images when a
pre-computed mask is not available. This function:

1. Smooths the image with a 2D Gaussian kernel
2. Runs `photutils` source detection above 3sigma
3. Dilates each footprint by 2 pixels to account for PSF wings

**Important:** Run this on both the reference and science images *before*
subtraction to prevent real source emission from leaking into the difference
image and contaminating the covariance estimate.


In [ ]:
# -- Visualize data and mask ----------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Difference image (masked pixels shown as black)
data_display = diff.copy()
data_display[mask] = np.nan
vmax = 3 * np.nanstd(data_display)
im0 = axes[0].imshow(data_display, origin='lower', cmap='bwr', vmin=-vmax, vmax=vmax)
axes[0].set_title('Difference Image (masked pixels removed)', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='Flux')

# Source mask
im1 = axes[1].imshow(mask, origin='lower', cmap='Reds', alpha=0.7)
axes[1].set_title(f'Source Mask ({mask.sum() / mask.size:.1%} covered)', fontsize=13)
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

fig.suptitle('F444W -- Difference image and source mask', y=1.02)
fig.tight_layout()
plt.show()

## 3. Find source-free squares

We search for the largest square region in the image that contains no sources.
The `find_zero_squares` function uses a summed-area table (integral image) for
efficient O(n^2) queries.

- **`a`**: minimum side length to search for (we use 60 pixels)
- **`max_nonzero`**: tolerance -- allow up to 5 "occupied" pixels (handles edge cases)


In [ ]:
# -- Find all source-free squares of size >= 60 ---------------------------
# Binary mask: 0 = free sky, 1 = occupied by source
occ = mask.astype(np.int64)
res_all = jpm.find_zero_squares(occ, a=60, all_sizes=True, max_nonzero=5)

print(f"Total source-free squares (>= 60 px): {len(res_all):,}")

# Sort by size
res_sorted = res_all[np.argsort(res_all[:, 2])]
size_max = res_sorted[-1, 2] if len(res_sorted) > 0 else 0
print(f"Largest square side length: {size_max} pixels")

In [ ]:
# -- Visualize the largest source-free squares ----------------------------
fig, ax = plt.subplots(figsize=(10, 10))

ax.imshow(diff, origin='lower', cmap='bwr', vmin=-vmax, vmax=vmax)
ax.set_title('Difference Image with Source-Free Squares', fontsize=14)
ax.axis('off')

# Draw the largest square
if len(res_sorted) > 0:
    top, left, size = int(res_sorted[-1, 0]), int(res_sorted[-1, 1]), int(res_sorted[-1, 2])
    from matplotlib.patches import Rectangle
    rect = Rectangle((left - 0.5, top - 0.5), size, size,
                     edgecolor='lime', facecolor='none', lw=2.5, zorder=100)
    ax.add_patch(rect)
    ax.text(left + size/2, top + size/2 + 10,
            f'Largest: {size}x{size}',
            ha='center', color='lime', fontsize=14, fontweight='bold',
            bbox=dict(boxstyle='round', fc='black', alpha=0.7))

fig.tight_layout()
plt.show()

## 4. Extract sky patch

We extract the largest source-free square and prepare it for kernel estimation:

1. Replace NaN/Inf pixels with Gaussian random noise at the local RMS
2. Subtract the sigma-clipped mean
3. Use the cleaned patch for autocorrelation computation


In [ ]:
# -- Extract the sky patch from the largest source-free square ------------
top, left, size = int(res_sorted[-1, 0]), int(res_sorted[-1, 1]), int(res_sorted[-1, 2])
patch = diff[top:top+size, left:left+size].copy()

# Replace NaN/Inf with Gaussian noise at local RMS
nan_mask = ~np.isfinite(patch)
if nan_mask.any():
    mean, median, std = sigma_clipped_stats(patch[np.isfinite(patch)], sigma=3)
    rng = np.random.RandomState(42)
    patch[nan_mask] = rng.randn(nan_mask.sum()) * std + mean
    patch = patch - mean

print(f'Sky patch shape: {patch.shape}')
_, _, std_patch = sigma_clipped_stats(patch, sigma=3)
print(f'Sky patch RMS: {std_patch:.4f}')

In [ ]:
# -- Visualize the sky patch ----------------------------------------------
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(patch, origin='lower', cmap='bwr', vmin=-3*std_patch, vmax=3*std_patch)
ax.set_title(f'Sky Patch ({size}x{size}) -- Top={top}, Left={left}', fontsize=13)
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Flux')
fig.tight_layout()
plt.show()

## 5. Estimate the covariance kernel

The covariance kernel is estimated from the sky patch via:

1. **Autocorrelation function (ACF)** -- `fftconvolve(patch, patch[::-1, ::-1])` computes
   the unnormalized ACF in Fourier space (O(N log N)).
2. **Normalization** -- divide by the peak value so the kernel center is 1.0.
3. **Central crop** -- extract the central `(size, size)` region.
4. **Cosine-bell window** -- taper the kernel edges to zero to prevent
   Gibbs ringing artefacts in the Fourier-domain likelihood.


In [ ]:
# -- Estimate covariance kernel -------------------------------------------
cov_kernel = jpm.estimate_cov_kernel(patch, size=15)

print(f'Kernel shape: {cov_kernel.shape}')
print(f'Kernel peak: {cov_kernel[cov_kernel.shape[0]//2, cov_kernel.shape[1]//2]:.4f}')
print(f'Kernel edge mean: {cov_kernel[[0, -1], :].mean():.6f}')

In [ ]:
# -- Visualize the covariance kernel --------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 2D kernel
im0 = axes[0].imshow(cov_kernel, origin='lower', cmap='viridis', vmin=0, vmax=1)
axes[0].set_title('Covariance Kernel (15x15)', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='Correlation')

# Radial profile (manual computation)
ny, nx = cov_kernel.shape
yv, xv = np.ogrid[:ny, :nx]
r = np.sqrt((xv - nx//2)**2 + (yv - ny//2)**2)
bin_edges = np.arange(0, nx//2 + 1)
radial_profile = np.zeros(len(bin_edges) - 1)
for i in range(len(bin_edges) - 1):
    mask_r = (r >= bin_edges[i]) & (r < bin_edges[i+1])
    if mask_r.any():
        radial_profile[i] = cov_kernel[mask_r].mean()

axes[1].plot(radial_profile, 'o-', color='C0', markersize=5, linewidth=1.5)
axes[1].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1].axhline(1/np.e, color='C1', linestyle=':', label='$1/e$ decay')
axes[1].set_xlabel('Radius (pixels)', fontsize=12)
axes[1].set_ylabel('Correlation Strength', fontsize=12)
axes[1].set_title('Radial Profile of Covariance Kernel', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].set_xlim(0, nx//2)

fig.suptitle('F444W -- Covariance kernel and radial profile', y=1.02)
fig.tight_layout()
plt.show()

## 6. Window function

A split-cosine-bell window tapers the kernel smoothly to zero at the edges.
Without windowing, the sharp cutoff would cause **Gibbs ringing** in the
Fourier domain -- artificial oscillations in the power spectrum that corrupt
the likelihood.

The window is defined by:
- **`alpha`**: taper width as a fraction of the kernel radius
- **`beta`**: flat-top inner radius as a fraction of the kernel radius


In [ ]:
# -- Visualize the window function ----------------------------------------
# Parameters from estimate_cov_kernel (size=15, r_in=2.0, r_out=6.0)
r_in, r_out = 2.0, 6.0
size_k = cov_kernel.shape[0]
alpha = (r_out - r_in) / ((size_k - 1) / 2)
beta = r_in / ((size_k - 1) / 2)
window = jpm.SplitCosineBellWindow(cov_kernel.shape, alpha=alpha, beta=beta)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(window, origin='lower', cmap='viridis', vmin=0, vmax=1)
axes[0].set_title(f'Window Function (alpha={alpha:.3f}, beta={beta:.3f})', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='Weight')

# Show windowed vs unwindowed kernel
axes[1].imshow(cov_kernel * window, origin='lower', cmap='viridis', vmin=0, vmax=1)
axes[1].set_title('Windowed Covariance Kernel', fontsize=13)
axes[1].axis('off')
plt.colorbar(im0, ax=axes[1], fraction=0.046, pad=0.04, label='Correlation')

fig.suptitle('Cosine-bell windowing prevents Fourier ringing', y=1.02)
fig.tight_layout()
plt.show()

## 7. Power spectrum

The power spectrum `P(k)` encodes the covariance structure in Fourier space.
It is computed as the real part of the FFT of the windowed kernel (shifted
to center with `ifftshift`). This is the term that enters the correlated-noise
log-likelihood at every MCMC step.

Low power at certain frequencies means those Fourier modes are **noisy** and
should be downweighted in the likelihood.


In [ ]:
# -- Compute the power spectrum -------------------------------------------
kernel_padded = np.zeros_like(patch)
ky, kx = cov_kernel.shape
ny, nx = patch.shape
y0, x0 = (ny - ky) // 2, (nx - kx) // 2
kernel_padded[y0:y0+ky, x0:x0+kx] = cov_kernel

kernel_shifted = np.fft.ifftshift(kernel_padded)
power_spectrum = np.real(np.fft.fft2(kernel_shifted))
power_spectrum = np.clip(power_spectrum, 1e-8, None)

print(f'Power spectrum range: [{power_spectrum.min():.2e}, {power_spectrum.max():.2e}]')

In [ ]:
# -- Visualize the power spectrum -----------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Log power spectrum
log_ps = np.log10(np.fft.fftshift(power_spectrum))
im0 = axes[0].imshow(log_ps, origin='lower', cmap='plasma')
axes[0].set_title('log10 Power Spectrum', fontsize=13)
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label='log10(P)')

# 1D power spectrum (azimuthally averaged)
yv_p, xv_p = np.ogrid[:ny, :nx]
r_p = np.sqrt((xv_p - nx//2)**2 + (yv_p - ny//2)**2)
bin_edges_p = np.logspace(-1, np.log10(nx//2), 30)
ps_1d = np.zeros(len(bin_edges_p) - 1)
for i in range(len(bin_edges_p) - 1):
    mask_r = (r_p >= bin_edges_p[i]) & (r_p < bin_edges_p[i+1])
    if mask_r.any():
        ps_1d[i] = power_spectrum[mask_r].mean()

axes[1].semilogy(bin_edges_p[:-1], ps_1d, 'o-', color='C0', markersize=4, linewidth=1.5)
axes[1].set_xlabel('Spatial frequency (cycles/px)', fontsize=12)
axes[1].set_ylabel('Power', fontsize=12)
axes[1].set_title('Azimuthally Averaged Power Spectrum', fontsize=13)
axes[1].grid(True, alpha=0.3)

fig.suptitle('F444W -- Power spectrum of the covariance kernel', y=1.02)
fig.tight_layout()
plt.show()

## 8. Pixel correlation whitening

This section demonstrates how the power spectrum removes pixel correlations:

1. **FFT** the sky patch -> `R(k)`
2. **Divide** by `sqrt(P(k))` -> whitened Fourier coefficients
3. **Inverse FFT** -> whitened real-space patch

After whitening, adjacent pixels should be uncorrelated (RMS should be
consistent with the local noise level).


In [ ]:
# -- Whitening pipeline ---------------------------------------------------
# FFT of the sky patch
R = np.fft.fft2(patch)

# Normalize by the power spectrum
P = np.abs(R)**2 / patch.size
P_safe = np.clip(P, 1e-10, None)

# Whitened Fourier coefficients
R_whitened = R / np.sqrt(P_safe)

# Inverse FFT to get the whitened patch
patch_whitened = np.real(np.fft.ifft2(R_whitened))

# Compare statistics
_, _, std_orig = sigma_clipped_stats(patch, sigma=3)
_, _, std_white = sigma_clipped_stats(patch_whitened, sigma=3)
print(f'Original patch RMS:    {std_orig:.4f}')
print(f'Whitened patch RMS:    {std_white:.4f}')

In [ ]:
# -- Visualize whitening --------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 0: Original patch
im0 = axes[0, 0].imshow(patch, origin='lower', cmap='bwr', vmin=-3*std_orig, vmax=3*std_orig)
axes[0, 0].set_title(f'Original Patch (RMS = {std_orig:.4f})', fontsize=12)
axes[0, 0].axis('off')
plt.colorbar(im0, ax=axes[0, 0], fraction=0.046, pad=0.04)

# Autocorrelation of original
acf_orig = jpm.estimate_cov_kernel(patch, size=15)
im1 = axes[0, 1].imshow(acf_orig, origin='lower', cmap='viridis', vmin=0, vmax=1)
axes[0, 1].set_title('ACF of Original Patch', fontsize=12)
axes[0, 1].axis('off')
plt.colorbar(im1, ax=axes[0, 1], fraction=0.046, pad=0.04)

# Radial profile of original ACF
ny_a, nx_a = acf_orig.shape
yv_a, xv_a = np.ogrid[:ny_a, :nx_a]
r_a = np.sqrt((xv_a - nx_a//2)**2 + (yv_a - ny_a//2)**2)
bin_edges_a = np.arange(0, nx_a//2 + 1)
rp_orig = np.zeros(len(bin_edges_a) - 1)
for i in range(len(bin_edges_a) - 1):
    mask_r = (r_a >= bin_edges_a[i]) & (r_a < bin_edges_a[i+1])
    if mask_r.any():
        rp_orig[i] = acf_orig[mask_r].mean()
axes[0, 2].semilogy(bin_edges_a[:-1], rp_orig, 'o-', color='C0', markersize=4)
axes[0, 2].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[0, 2].set_xlabel('Radius (px)')
axes[0, 2].set_ylabel('Correlation')
axes[0, 2].set_title('Radial Profile of Original ACF', fontsize=12)
axes[0, 2].grid(True, alpha=0.3)

# Row 1: Whitened patch
im2 = axes[1, 0].imshow(patch_whitened, origin='lower', cmap='bwr', vmin=-3*std_white, vmax=3*std_white)
axes[1, 0].set_title(f'Whitened Patch (RMS = {std_white:.4f})', fontsize=12)
axes[1, 0].axis('off')
plt.colorbar(im2, ax=axes[1, 0], fraction=0.046, pad=0.04)

# ACF of whitened
acf_white = jpm.estimate_cov_kernel(patch_whitened, size=15)
im3 = axes[1, 1].imshow(acf_white, origin='lower', cmap='viridis', vmin=-0.1, vmax=1)
axes[1, 1].set_title('ACF of Whitened Patch', fontsize=12)
axes[1, 1].axis('off')
plt.colorbar(im3, ax=axes[1, 1], fraction=0.046, pad=0.04)

# Radial profile of whitened ACF
ny_aw, nx_aw = acf_white.shape
yv_aw, xv_aw = np.ogrid[:ny_aw, :nx_aw]
r_aw = np.sqrt((xv_aw - nx_aw//2)**2 + (yv_aw - ny_aw//2)**2)
bin_edges_aw = np.arange(0, nx_aw//2 + 1)
rp_white = np.zeros(len(bin_edges_aw) - 1)
for i in range(len(bin_edges_aw) - 1):
    mask_r = (r_aw >= bin_edges_aw[i]) & (r_aw < bin_edges_aw[i+1])
    if mask_r.any():
        rp_white[i] = acf_white[mask_r].mean()
axes[1, 2].semilogy(bin_edges_aw[:-1], rp_white, 'o-', color='C1', markersize=4)
axes[1, 2].axhline(0, color='k', linestyle='--', alpha=0.5)
axes[1, 2].set_xlabel('Radius (px)')
axes[1, 2].set_ylabel('Correlation')
axes[1, 2].set_title('Radial Profile of Whitened ACF', fontsize=12)
axes[1, 2].grid(True, alpha=0.3)

fig.suptitle('F444W -- Pixel correlation whitening demonstration', y=1.01)
fig.tight_layout()
plt.show()

## Summary

| Quantity | Value |
|----------|-------|
| Image size | 401 x 401 px |
| Source coverage | See notebook output |
| Largest source-free square | See notebook output |
| Sky patch RMS | See notebook output |
| Covariance kernel size | 15 x 15 px |
| Kernel peak (center) | 1.0000 |
| Whitened patch RMS | See notebook output |

The covariance kernel captures the spatially correlated noise introduced by the
drizzle algorithm. By encoding this structure as a Fourier-space power spectrum,
`jwst_psfmc` enables accurate flux uncertainty estimation that accounts for
pixel correlations -- critical for reliable transient photometry on JWST data.

---

*Notebook generated with `jwst_psfmc` -- see [README](../README.md) for installation.*